# 14v1 - A2 v3: window-only fold-0 pilot

**Plan item:** tests whether A3 v2's widened per-plane slice-sampling
window (built by `13v1`, no `src/` changes) improves the weak-finding
cluster on A2 v2's own architecture. Full design, evidence, and the
gate this run is measured against:
docs/superpowers/specs/2026-08-31-a3v2-window-widening-design.md
(approved after one Opus review pass).

Self-contained per this project's Kaggle constraint (no `import src`) -
identical to `09v1`'s own labels/folds/model/eval code (A2 v2's winning
architecture, unchanged) except `CACHE_DIR` points at `13v1`'s new
cache and the checkpoint-selection cells below (dual best-epoch/SWA
readout, per the spec).

**Scope: fold 0 only, window width is the only changed variable** vs.
A2 v2 (`09v1_a2v2_multigroup_baseline.ipynb`) - same hyperparameters,
same labels, same fold split, same 224px/130mm crop.

In [ ]:
import hashlib
import re
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

_KAGGLE_RAW = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
ON_KAGGLE = _KAGGLE_RAW.exists()
if not ON_KAGGLE:
    raise RuntimeError(
        "This notebook needs the full DICOM tree + GPU + the A3 v2 cache "
        "attached - run on Kaggle, not locally."
    )
RAW_DIR = _KAGGLE_RAW
# EDIT this path once 13v1's cache Dataset is uploaded and attached - this
# is NOT the current (narrow-window) cache, it's the new widened-window
# one from 13v1_a3v2_window_cache_build.ipynb.
CACHE_DIR = Path("/kaggle/input/datasets/alherma7/cache-a3v2-window-wide/cache")

# llm_labels_v4_blend.csv is NOT part of the official competition mount -
# it's the user's own separately-attached Kaggle Dataset, unchanged from
# A2 v2. EDIT this path to match wherever it actually lands under
# /kaggle/input/ once attached (check `!ls /kaggle/input` if unsure).
PUBLISHED_LABELS_PATH = Path("/kaggle/input/llm-labels-v4-blend/llm_labels_v4_blend.csv")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
print("CACHE_DIR:", CACHE_DIR, "exists:", CACHE_DIR.exists())
print("PUBLISHED_LABELS_PATH:", PUBLISHED_LABELS_PATH, "exists:", PUBLISHED_LABELS_PATH.exists())

FINDINGS = [
    "acl_injury", "mcl_injury", "medial_meniscus_tear", "lateral_meniscus_tear",
    "oa_medial_compartment", "oa_lateral_compartment", "oa_patellofemoral_compartment",
    "effusion", "synovitis", "bakers_cyst", "bone_contusion", "fracture",
]
OFFICIAL_LABEL_COLUMNS = {
    "acl_injury": "ACL", "mcl_injury": "MCL",
    "medial_meniscus_tear": "Medial Meniscus", "lateral_meniscus_tear": "Lateral Meniscus",
    "oa_medial_compartment": "Medial OA", "oa_lateral_compartment": "Lateral OA",
    "oa_patellofemoral_compartment": "PF OA", "effusion": "Effusion",
    "synovitis": "Synovitis", "bakers_cyst": "Baker's",
    "bone_contusion": "Contusion", "fracture": "Fracture",
}
SLOT_NAMES = ["SAG_FLUID_FS", "COR_FLUID_FS", "AX_FLUID_FS", "SAG_FLUID_NOFS", "COR_T1", "SAG_T1"]
SLOT_CACHE_GROUP_SIZE = 3
SLOT_CACHE_N_GROUPS = 3
N_SLOTS = len(SLOT_NAMES) * SLOT_CACHE_N_GROUPS  # 18 pseudo-slots -- A2 v2's own architecture, unchanged
CV_FOLDS = 4
TRAIN_SHARDS = [f"train.s{i:02d}of04" for i in range(4)]

## Labels: gold official values + A1a' published set for weak studies

In [ ]:
def load_published_labels(path):
    published = pd.read_csv(path)
    label_cols = list(OFFICIAL_LABEL_COLUMNS.values())
    published = published.set_index("StudyInstanceUID")[label_cols]
    published.columns = list(OFFICIAL_LABEL_COLUMNS.keys())
    return published


def load_gold_labels(raw_dir):
    train = pd.read_csv(raw_dir / "train.csv")
    label_cols = list(OFFICIAL_LABEL_COLUMNS.values())
    gold_mask = train[label_cols].notna().all(axis=1)
    gold = train.loc[gold_mask, ["StudyInstanceUID"] + label_cols].set_index("StudyInstanceUID")
    gold.columns = list(OFFICIAL_LABEL_COLUMNS.keys())
    return gold


train_csv = pd.read_csv(RAW_DIR / "train.csv")
reports = train_csv.set_index("StudyInstanceUID")[["Report"]]
gold = load_gold_labels(RAW_DIR)
published = load_published_labels(PUBLISHED_LABELS_PATH)

missing = set(train_csv["StudyInstanceUID"]) - set(published.index)
print("train.csv studies missing from published labels:", len(missing))
assert len(missing) == 0

is_gold = reports.index.isin(gold.index)
label_table = published.reindex(reports.index)[FINDINGS].copy()
label_table.loc[gold.index, FINDINGS] = gold[FINDINGS]
label_table["is_gold"] = is_gold
print(label_table.shape, "gold rows:", label_table["is_gold"].sum())

## Folds: report-template + scanner-fingerprint grouping (A0)

Identical logic to `05v2`/`06v2`/`09v1`'s fold-assignment cell -
`GroupKFold` has no shuffling/randomness, so recomputing from the same
inputs reproduces the exact same split. **Asserted, not just assumed**
- fold assignment is independent of the cache rebuild (it comes from
report/scanner grouping, not slice windowing), but this notebook is a
fresh execution, so the assertion below is re-run rather than trusted
by proximity to `09v1`'s own passing result.

In [ ]:
def report_group_key(report_text):
    if not isinstance(report_text, str):
        normalized = ""
    else:
        t = unicodedata.normalize("NFKD", report_text.lower())
        t = "".join(ch for ch in t if not unicodedata.combining(ch))
        normalized = re.sub(r"\s+", " ", t).strip()
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


SCANNER_FINGERPRINT_TAGS = (
    "Manufacturer", "ManufacturerModelName", "InstitutionName",
    "DeviceSerialNumber", "MagneticFieldStrength", "StationName",
)


def build_scanner_fingerprints(raw_dir, split="train"):
    series = pd.read_csv(raw_dir / f"{split}_series.csv")
    first_series = series.drop_duplicates("StudyInstanceUID", keep="first")
    fingerprints = {}
    for row in first_series.itertuples(index=False):
        series_dir = raw_dir / f"{split}_series" / row.StudyInstanceUID / row.SeriesInstanceUID
        files = sorted(series_dir.glob("*.dcm"))
        if not files:
            fingerprints[row.StudyInstanceUID] = None
            continue
        ds = pydicom.dcmread(files[0], stop_before_pixels=True)
        fingerprints[row.StudyInstanceUID] = tuple(
            str(getattr(ds, tag, None)) for tag in SCANNER_FINGERPRINT_TAGS
        )
    result = pd.Series(fingerprints, name="scanner_fingerprint")
    result.index.name = "StudyInstanceUID"
    return result


def build_group_ids(*group_key_series):
    index = group_key_series[0].index
    parent = {i: i for i in index}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    for keys in group_key_series:
        valid = keys.dropna()
        for _, idx in valid.groupby(valid).groups.items():
            idx = list(idx)
            for other in idx[1:]:
                union(idx[0], other)

    return pd.Series({i: find(i) for i in index}, name="group_id")


t0 = time.time()
scanner_fp = build_scanner_fingerprints(RAW_DIR, split="train")
print(f"scanner fingerprints: {time.time() - t0:.1f}s for {len(scanner_fp)} studies")

group_keys = reports["Report"].apply(report_group_key)
group_ids = build_group_ids(group_keys, scanner_fp.reindex(reports.index))

gkf = GroupKFold(n_splits=CV_FOLDS)
fold = pd.Series(-1, index=reports.index, dtype=int)
for fold_idx, (_, val_idx) in enumerate(gkf.split(reports, groups=group_ids.to_numpy())):
    fold.iloc[val_idx] = fold_idx
label_table["fold"] = fold
print(label_table["fold"].value_counts().sort_index())

fold0_val = label_table[label_table["fold"] == 0]
print(f"\nfold 0 val: {len(fold0_val)} studies, {int(fold0_val['is_gold'].sum())} gold")
assert len(fold0_val) == 1307, (
    f"fold 0 val set size {len(fold0_val)} != recorded 1307 - fold split diverged, "
    "stop and investigate before trusting any comparison against 0.7956"
)
assert int(fold0_val["is_gold"].sum()) == 17, (
    f"fold 0 gold count {int(fold0_val['is_gold'].sum())} != recorded 17"
)
print("fold 0 matches the recorded split exactly - OK, safe to compare against 0.7956")

## Reshape: `select_group()` (A3, unchanged) + `expand_slot_groups()` (A2 v2, unchanged)

Both unchanged from `09v1`/`10v1` - the cache's shape doesn't change
between the narrow and widened window (spec section 1), so this
reshape logic has no dependency on which physical slices the pixels
came from.

In [ ]:
def select_group(cache_slot_stack, group_index):
    if isinstance(group_index, int):
        group_index = [group_index]
    size = SLOT_CACHE_GROUP_SIZE
    groups = [cache_slot_stack[..., g * size:(g + 1) * size, :, :] for g in group_index]
    return np.concatenate(groups, axis=-3)


def expand_slot_groups(cache_slot_stack, slot_mask):
    '''cache_slot_stack: (n_slots, 9, H, W) - one study, all slots.
    slot_mask: (n_slots,). Returns (images, mask):
    images (n_slots * SLOT_CACHE_N_GROUPS, SLOT_CACHE_GROUP_SIZE, H, W),
    slot-major order (pseudo-slot index = s * SLOT_CACHE_N_GROUPS + g).
    mask (n_slots * SLOT_CACHE_N_GROUPS,), each real slot's bit repeated
    SLOT_CACHE_N_GROUPS times.'''
    n_slots = cache_slot_stack.shape[0]
    h, w = cache_slot_stack.shape[-2:]
    groups = [select_group(cache_slot_stack, g) for g in range(SLOT_CACHE_N_GROUPS)]
    stacked = np.stack(groups, axis=1)  # (n_slots, n_groups, 3, H, W)
    images = stacked.reshape(n_slots * SLOT_CACHE_N_GROUPS, SLOT_CACHE_GROUP_SIZE, h, w)
    mask = np.repeat(slot_mask, SLOT_CACHE_N_GROUPS)
    return images, mask


# Self-validate: each channel's pixels are set to that channel's global
# index, so the expected output is exact and checkable by hand.
_demo_stack = np.zeros((6, 9, 4, 4), dtype=np.uint8)
for _c in range(9):
    _demo_stack[:, _c] = _c
_demo_mask = np.array([1.0, 0.0, 1.0, 1.0, 0.0, 1.0], dtype=np.float32)
_images, _mask = expand_slot_groups(_demo_stack, _demo_mask)
assert _images.shape == (18, 3, 4, 4)
assert _mask.shape == (18,)
for _s in range(6):
    for _g in range(3):
        assert np.array_equal(_images[_s * 3 + _g], select_group(_demo_stack[_s], _g))
assert np.array_equal(_mask, np.repeat(_demo_mask, 3))
print("expand_slot_groups matches select_group per (slot, group) pair and replicates the mask - OK")

## Cache dataset

Opens all 4 train shards as memmaps (never materialises the full
cache in RAM). Unchanged from `09v1`/`10v1`.

In [ ]:
class SlotCacheDataset(torch.utils.data.Dataset):
    def __init__(self, cache_dir, shards, labels_df, group_index=1, expand_groups=False, study_ids=None):
        if expand_groups and group_index != 1:
            raise ValueError(
                "expand_groups=True ignores group_index; pass group_index=1 (default) or omit "
                f"it, not group_index={group_index!r}"
            )
        self.group_index = group_index
        self.expand_groups = expand_groups
        caches, masks, all_study_ids, shard_of, local_idx = [], [], [], [], []
        for shard in shards:
            cache = np.load(cache_dir / f"{shard}_cache.npy", mmap_mode="r")
            mask = np.load(cache_dir / f"{shard}_mask.npy")
            studies = pd.read_csv(cache_dir / f"{shard}_studies.csv")
            caches.append(cache)
            masks.append(mask)
            all_study_ids.append(studies["StudyInstanceUID"].to_numpy())
            shard_of.append(np.full(len(studies), len(caches) - 1))
            local_idx.append(np.arange(len(studies)))

        self.caches = caches
        mask_all = np.concatenate(masks, axis=0).astype(np.float32)
        study_ids_all = np.concatenate(all_study_ids)
        shard_of_all = np.concatenate(shard_of)
        local_idx_all = np.concatenate(local_idx)

        if study_ids is not None:
            keep = np.isin(study_ids_all, np.asarray(list(study_ids)))
            mask_all, study_ids_all = mask_all[keep], study_ids_all[keep]
            shard_of_all, local_idx_all = shard_of_all[keep], local_idx_all[keep]

        self.mask = mask_all
        self.study_ids = study_ids_all
        self.shard_of = shard_of_all
        self.local_idx = local_idx_all

        aligned = labels_df.reindex(self.study_ids)[FINDINGS]
        if aligned.isna().any().any():
            missing = self.study_ids[aligned.isna().any(axis=1).to_numpy()]
            raise ValueError(f"{len(missing)} cache studies missing labels, e.g. {missing[:5]}")
        self.labels = aligned.to_numpy(dtype=np.float32)

    def __len__(self):
        return len(self.study_ids)

    def __getitem__(self, i):
        shard_idx, row = self.shard_of[i], self.local_idx[i]
        full = self.caches[shard_idx][row]  # (6, 9, 224, 224) uint8
        if self.expand_groups:
            selected, slot_mask_row = expand_slot_groups(full, self.mask[i])
        else:
            g = self.group_index
            selected = full[:, g * SLOT_CACHE_GROUP_SIZE:(g + 1) * SLOT_CACHE_GROUP_SIZE]
            slot_mask_row = self.mask[i]
        images = torch.from_numpy(np.ascontiguousarray(selected)).float() / 255.0
        mask = torch.from_numpy(slot_mask_row)
        label = torch.from_numpy(self.labels[i])
        return images, mask, label


sanity_ds = SlotCacheDataset(CACHE_DIR, TRAIN_SHARDS[:1], label_table, expand_groups=True)
images, mask, label = sanity_ds[0]
print("images:", images.shape, images.dtype, "mask:", mask.shape, "label:", label.shape)
assert images.shape == (N_SLOTS, 3, 224, 224)
assert mask.shape == (N_SLOTS,)
assert not torch.isnan(images).any()
print("SlotCacheDataset(expand_groups=True) sanity check OK")

try:
    SlotCacheDataset(CACHE_DIR, TRAIN_SHARDS[:1], label_table, expand_groups=True, group_index=0)
    raise AssertionError("expand_groups=True with a non-default group_index should have raised")
except ValueError:
    print("expand_groups=True with group_index=0 correctly raises - OK")

In [ ]:
# Cache-identity guard (spec section 3, first-review finding I5).
import json as _json

_meta_path = CACHE_DIR / "cache_meta.json"
print("cache_meta.json:", _meta_path, "exists:", _meta_path.exists())
assert _meta_path.exists(), "cache_meta.json not found - check CACHE_DIR points at the 13v1 output"
_meta = _json.loads(_meta_path.read_text())
print("cache fingerprint:", _json.dumps(_meta, indent=2, sort_keys=True))

_EXPECTED = {"window": "default", "crop_mm": 130.0, "img": 224, "group": 3, "n_group": 3}
_mismatch = {k: (v, _meta.get(k)) for k, v in _EXPECTED.items() if _meta.get(k) != v}
assert not _mismatch, (
    f"cache_meta.json doesn't match this notebook's expectations: {_mismatch} - "
    "CACHE_DIR may point at the wrong (or a stale) Dataset. This must be the widened-"
    "window cache from 13v1, not the current narrow-window one."
)
_expected_slots = ["SAG_FLUID_FS", "COR_FLUID_FS", "AX_FLUID_FS", "SAG_FLUID_NOFS", "COR_T1", "SAG_T1"]
assert _meta.get("slots") == _expected_slots, f"slot order mismatch: {_meta.get('slots')}"
print("cache-identity guard passed - window='default' (widened), crop_mm/img/group/n_group/slots all match")

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "timm"], check=True)
import timm
print("timm:", timm.__version__)

## Model: DINOv2-small backbone + masked_finding_attention

Unchanged from A2 v2 (`masked_finding_attention()`/`SlotAttentionModel`
are already generic over slot count - zero code changes needed) except
`n_slots` now defaults to `N_SLOTS` (18) instead of 6, same as `09v1`.

In [ ]:
def masked_finding_attention(embeddings, mask, query, head_weight, head_bias):
    if not (mask.sum(dim=1) > 0).all():
        raise ValueError("masked_finding_attention: a row has 0 present slots")
    scores = torch.einsum("od,bsd->bos", query, embeddings) / (embeddings.shape[-1] ** 0.5)
    expanded_mask = mask.unsqueeze(1).expand(-1, query.shape[0], -1)
    scores = scores.masked_fill(expanded_mask == 0, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    context = torch.einsum("bos,bsd->bod", weights, embeddings)
    logits = (context * head_weight.unsqueeze(0)).sum(-1) + head_bias
    if torch.isnan(logits).any() or torch.isinf(logits).any():
        raise RuntimeError("masked_finding_attention produced NaN/Inf logits")
    return logits


class SlotAttentionModel(nn.Module):
    def __init__(self, n_findings=len(FINDINGS), n_slots=N_SLOTS,
                 backbone_name="vit_small_patch14_dinov2.lvd142m", unfreeze_last=6):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, num_classes=0, img_size=224,
        )
        embed_dim = self.backbone.num_features
        for p in self.backbone.parameters():
            p.requires_grad = False
        for block in self.backbone.blocks[-unfreeze_last:]:
            for p in block.parameters():
                p.requires_grad = True

        self.query = nn.Parameter(torch.randn(n_findings, embed_dim) * (embed_dim ** -0.5))
        self.heads = nn.Linear(embed_dim, n_findings)
        self.embed_dim = embed_dim
        self.n_findings = n_findings
        self.n_slots = n_slots

    def forward(self, slot_images, slot_mask):
        B, S, C, H, W = slot_images.shape
        if (S, C, H, W) != (self.n_slots, 3, 224, 224):
            raise ValueError(
                f"expected slot_images (*, {self.n_slots}, 3, 224, 224), got {tuple(slot_images.shape)}"
            )
        if tuple(slot_mask.shape) != (B, S):
            raise ValueError(f"expected slot_mask ({B}, {S}), got {tuple(slot_mask.shape)}")

        flat = slot_images.view(B * S, C, H, W)
        embeddings = self.backbone(flat).view(B, S, self.embed_dim)
        return masked_finding_attention(
            embeddings, slot_mask, self.query, self.heads.weight, self.heads.bias
        )


print("SlotAttentionModel defined (n_slots=18) - instantiating to confirm it loads real DINOv2 weights...")
_smoke_model = SlotAttentionModel()
n_trainable = sum(p.numel() for p in _smoke_model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in _smoke_model.parameters())
print(f"embed_dim={_smoke_model.embed_dim}, n_slots={_smoke_model.n_slots}, "
      f"trainable params={n_trainable:,} / {n_total:,}")
del _smoke_model

## Evaluation helpers

Unchanged hand-kept copies of `src/evaluate.py::macro_roc_auc`/`per_finding_roc_auc`.

In [ ]:
def per_finding_roc_auc(y_true, y_pred):
    scores = {}
    for c in y_true.columns:
        if y_true[c].nunique() < 2:
            scores[c] = float("nan")
        else:
            scores[c] = roc_auc_score(y_true[c], y_pred[c])
    return pd.Series(scores)


def macro_roc_auc(y_true, y_pred):
    per_finding = per_finding_roc_auc(y_true, y_pred)
    undefined = per_finding[per_finding.isna()]
    if len(undefined) > 0:
        print(f"  (macro_roc_auc: {len(undefined)} finding(s) undefined this fold "
              f"- {list(undefined.index)}, excluded from the mean, not treated as 0)")
    return float(per_finding.mean())

## Pre-flight: overfit 8 real studies, check VRAM/host-RAM headroom

Standard wiring sanity check (same practice as A2 v1/A2 v2) plus the
VRAM/host-RAM read, since 18 pseudo-slots costs ~3x the memory of a
6-slot input per study.

In [ ]:
model = SlotAttentionModel().to(DEVICE)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                         lr=1e-3, weight_decay=0.02)

tiny_studies = label_table.index[:8]
tiny_ds = SlotCacheDataset(CACHE_DIR, TRAIN_SHARDS, label_table, expand_groups=True, study_ids=tiny_studies)
tiny_loader = torch.utils.data.DataLoader(tiny_ds, batch_size=8, shuffle=False)
images, mask, labels = next(iter(tiny_loader))
images, mask, labels = images.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
bytes_per_batch = images.element_size() * images.nelement()
print(f"images tensor: {tuple(images.shape)}, {bytes_per_batch / 1e6:.1f} MB for this "
      f"batch of {images.shape[0]} studies at 18 pseudo-slots - host-RAM sanity check.")

eps = 1e-7
loss_floor = -(labels * torch.log(labels.clamp(eps, 1)) +
               (1 - labels) * torch.log((1 - labels).clamp(eps, 1))).mean().item()
print(f"loss floor for this batch (soft-label entropy): {loss_floor:.4f}")

print("pre-flight: overfitting 8 real studies...")
model.train()
for step in range(500):
    opt.zero_grad()
    loss = F.binary_cross_entropy_with_logits(model(images, mask), labels)
    loss.backward()
    opt.step()
    if step % 100 == 0:
        print(f"  step {step}: loss={loss.item():.4f}")
print(f"final pre-flight loss: {loss.item():.4f} (floor: {loss_floor:.4f})")
assert loss.item() < loss_floor + 0.02, (
    f"model failed to reach the soft-label loss floor ({loss_floor:.4f}) on 8 real "
    f"studies - stop and debug before the real run"
)
if DEVICE.type == "cuda":
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    print(f"peak VRAM during pre-flight (batch=8): {peak_gb:.2f} GB - use this to judge "
          "whether MICRO_BATCH=8/ACCUMULATE_STEPS=4 (A2 v2's own measured-safe split) "
          "still fits, or needs adjusting.")
print("pre-flight OK")
del model, opt

## Full fold-0 training run - dual checkpoint tracking (spec section 3)

Hyperparameters unchanged from A2 v2. Tracks **both** a best-epoch
checkpoint (matching every prior notebook's `if gold_auc > best_gold_auc`
selection - this is what gates against the A2 v2 baseline,
apples-to-apples) **and** an SWA-averaged checkpoint (this project's
first adoption of [[feedback-checkpoint-selection-noise]]'s fix,
averaging the last 3 of 12 epochs' weights instead of selecting one).
Both mechanisms coexist in the same training loop, per the reference
kernel's own `train_fold` precedent.

In [ ]:
SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)
# 09v1 (the A2 v2 baseline this pilot is gated against) was never
# seeded - this is a real, accepted limitation, not silently fixed
# retroactively (spec section 3, M4). Documented here, not hidden.

FOLD = 0
train_idx = np.flatnonzero(label_table["fold"].to_numpy() != FOLD)
val_idx = np.flatnonzero(label_table["fold"].to_numpy() == FOLD)
val_labels = label_table.iloc[val_idx].reset_index()
val_is_gold = val_labels["is_gold"].to_numpy()
print(f"fold {FOLD}: {len(train_idx)} train / {len(val_idx)} val ({val_is_gold.sum()} gold in val)")

BATCH_SIZE = 32
MICRO_BATCH = 8
ACCUMULATE_STEPS = BATCH_SIZE // MICRO_BATCH
assert BATCH_SIZE % MICRO_BATCH == 0, "MICRO_BATCH must evenly divide BATCH_SIZE"
print(f"BATCH_SIZE={BATCH_SIZE}, MICRO_BATCH={MICRO_BATCH}, ACCUMULATE_STEPS={ACCUMULATE_STEPS}")

full_ds = SlotCacheDataset(CACHE_DIR, TRAIN_SHARDS, label_table, expand_groups=True)
train_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(full_ds, train_idx.tolist()), batch_size=MICRO_BATCH,
    shuffle=True, num_workers=2, drop_last=True,
)
val_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(full_ds, val_idx.tolist()), batch_size=MICRO_BATCH, shuffle=False, num_workers=2,
)

model = SlotAttentionModel().to(DEVICE)
backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("backbone")]
head_params = [p for n, p in model.named_parameters() if not n.startswith("backbone")]
opt = torch.optim.AdamW([
    {"params": backbone_params, "lr": 8e-6},
    {"params": head_params, "lr": 1e-3},
], weight_decay=0.02)

EPOCHS = 12
# SWA_EPOCHS=3: averaging the last 3 of 12 epochs. Under this notebook's
# own OneCycleLR (no pct_start passed -> PyTorch default 0.3, LR peaks at
# epoch ~3.6), epochs 10/11/12 sit at ~28.3%/13.3%/3.5% of max_lr -
# monotonically annealing, well past the schedule's peak, no warmup
# contamination (spec section 3, M2). Not a sourced value - the
# reference kernel exposes RSNA_SWA_EPOCHS but states no chosen number
# anywhere in its own file.
SWA_EPOCHS = 3

steps_per_epoch = len(train_loader) // ACCUMULATE_STEPS
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=[8e-6, 1e-3], total_steps=EPOCHS * steps_per_epoch
)

best_gold_auc = -1.0
best_state = None
swa_state, swa_n = None, 0
for epoch in range(EPOCHS):
    model.train()
    t0 = time.time()
    opt.zero_grad()
    for step, (images, mask, labels) in enumerate(train_loader):
        images, mask, labels = images.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
        loss = F.binary_cross_entropy_with_logits(model(images, mask), labels) / ACCUMULATE_STEPS
        loss.backward()
        if (step + 1) % ACCUMULATE_STEPS == 0:
            opt.step()
            opt.zero_grad()
            scheduler.step()

    model.eval()
    probs = []
    with torch.no_grad():
        for images, mask, _ in val_loader:
            images, mask = images.to(DEVICE), mask.to(DEVICE)
            probs.append(torch.sigmoid(model(images, mask)).cpu().numpy())
    val_pred = pd.DataFrame(np.concatenate(probs), columns=FINDINGS)
    gold_auc = macro_roc_auc(val_labels.loc[val_is_gold, FINDINGS], val_pred[val_is_gold])
    print(f"epoch {epoch}: {time.time() - t0:.0f}s, val gold macro-AUC={gold_auc:.4f}")

    if gold_auc > best_gold_auc:
        best_gold_auc = gold_auc
        # .clone() is load-bearing (spec section 3, I7): .cpu() is a
        # no-op on a tensor already on CPU, so without .clone() best_state
        # would alias the live model parameters and the NEXT epoch's
        # training would silently corrupt this saved copy.
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print("  new best (epoch-selected)")

    if epoch >= EPOCHS - SWA_EPOCHS:
        # Same .clone() hazard applies here, for the same reason -
        # explicit per spec section 3, I7.
        sd = {k: v.detach().float().cpu().clone() for k, v in model.state_dict().items()}
        if swa_state is None:
            swa_state, swa_n = sd, 1
        else:
            swa_n += 1
            for k in swa_state:
                swa_state[k] += (sd[k] - swa_state[k]) / swa_n
        print(f"  SWA: accumulated epoch {epoch} (swa_n={swa_n})")

assert best_state is not None and swa_state is not None
torch.save(best_state, "/kaggle/working/a2_v3_fold0_best_epoch.pt")
torch.save(swa_state, "/kaggle/working/a2_v3_fold0_swa.pt")
print(f"\nbest-epoch gold macro-AUC (fold {FOLD}, selection-based): {best_gold_auc:.4f}")
print(f"SWA checkpoint: averaged {swa_n} epochs, no selection - gold AUC computed next cell")

## Final report: dual readout (best-epoch + SWA), spec section 4 gate

Evaluates **both** checkpoints on fold 0's gold subset - the reference
kernel's own SWA branch only ever scores derived/weak labels for the
averaged model, never gold, so this gold-AUC-for-SWA readout is built
here, not copied. The gate itself (spec section 4) runs on the
**best-epoch** number, apples-to-apples with how A2 v2's own 0.7956
baseline was produced. `mcl_injury` is reported as context only, per
the spec's corrected reasoning - it does not gate. The decision rule is
pre-agreed (spec section 4): macro passes and **at least one** of the
two meniscus findings clears the noise bar -> scale.

In [ ]:
def eval_gold(state_dict, label):
    m = SlotAttentionModel().to(DEVICE)
    m.load_state_dict(state_dict)
    m.eval()
    probs = []
    with torch.no_grad():
        for images, mask, _ in val_loader:
            images, mask = images.to(DEVICE), mask.to(DEVICE)
            probs.append(torch.sigmoid(m(images, mask)).cpu().numpy())
    pred = pd.DataFrame(np.concatenate(probs), columns=FINDINGS)
    per_finding = per_finding_roc_auc(val_labels.loc[val_is_gold, FINDINGS], pred[val_is_gold])
    macro_11 = float(per_finding.drop(index="oa_lateral_compartment").mean())
    print(f"\n{label} gold per-finding AUC:\n{per_finding}")
    print(f"{label} gold macro (11-finding): {macro_11:.4f}")
    del m
    return per_finding, macro_11


gold_per_finding_best, candidate_macro_best = eval_gold(best_state, "best-epoch")
gold_per_finding_swa, candidate_macro_swa = eval_gold(swa_state, "SWA")

print(f"\nselection-noise gap (best-epoch minus SWA): {candidate_macro_best - candidate_macro_swa:+.4f} "
      "- this project's first real, on-our-own-data measurement of this gap "
      "(see feedback_checkpoint_selection_noise.md)")

# ---- spec section 4: fold-0 pilot gate, on the BEST-EPOCH number (apples-to-apples) ----
BASELINE_FOLD0_MACRO_11 = 0.7956  # A2 v2 fold 0, 11-finding mean (09v1's real output)
BASELINE_MEDIAL_MENISCUS = 0.597
BASELINE_LATERAL_MENISCUS = 0.652
GOLD_TOL = 0.03

macro_delta = candidate_macro_best - BASELINE_FOLD0_MACRO_11
macro_ok = macro_delta >= -GOLD_TOL
print(f"\ngold macro (best-epoch, 11-finding) vs. A2 v2 fold-0 baseline {BASELINE_FOLD0_MACRO_11}: "
      f"delta={macro_delta:+.4f}, macro_ok={macro_ok} (tol={GOLD_TOL})")

medial_delta = float(gold_per_finding_best["medial_meniscus_tear"]) - BASELINE_MEDIAL_MENISCUS
lateral_delta = float(gold_per_finding_best["lateral_meniscus_tear"]) - BASELINE_LATERAL_MENISCUS
print(f"\nmedial_meniscus_tear: {gold_per_finding_best['medial_meniscus_tear']:.4f} "
      f"(baseline {BASELINE_MEDIAL_MENISCUS}, delta={medial_delta:+.4f})")
print(f"lateral_meniscus_tear: {gold_per_finding_best['lateral_meniscus_tear']:.4f} "
      f"(baseline {BASELINE_LATERAL_MENISCUS}, delta={lateral_delta:+.4f})")

# mcl_injury: reported as context only, does NOT gate (spec section 4,
# first-review finding I3 - its A2v2 fold-0 baseline is 0.800, and this
# specific change is not provably a no-op for it the way A2v2's own
# group-recombination change was).
BASELINE_MCL = 0.800
mcl_delta = float(gold_per_finding_best["mcl_injury"]) - BASELINE_MCL
print(f"\nmcl_injury (context only, not gating): {gold_per_finding_best['mcl_injury']:.4f} "
      f"(baseline {BASELINE_MCL}, delta={mcl_delta:+.4f})")

# Pre-agreed decision rule (spec section 4, first-review finding I4,
# user's explicit choice) - "at least one" meniscus finding, not "both",
# fixed in advance rather than relitigated after seeing an ambiguous
# result the way A2 v2's own pilot had to be.
at_least_one_real = medial_delta >= 0.10 or lateral_delta >= 0.10
print(f"\nat least one meniscus finding moved >=0.10 (pre-agreed bar): {at_least_one_real}")

print()
if not macro_ok:
    print("DECISION: macro regressed beyond tolerance -> STOP, do not scale to 4 folds.")
elif at_least_one_real:
    print("DECISION: macro OK and at least one meniscus finding moved meaningfully positive "
          "-> SCALE to the remaining 3 folds (pre-agreed rule, spec section 4).")
else:
    print("DECISION: macro OK but neither meniscus finding cleared the noise bar -> STOP, "
          "report as inconclusive, NOT disproved.")

print(f"\nSWA gold macro (11-finding, forward-looking figure): {candidate_macro_swa:.4f}")
print("\nfull gold per-finding AUC (best-epoch) for the record:\n", gold_per_finding_best)
print("\nfull gold per-finding AUC (SWA) for the record:\n", gold_per_finding_swa)

## Real output (fold-0 pilot, real Kaggle run, reported 2026-08-31)

Only the final dual-checkpoint report + DECISION line were pasted back
this run - the fold-assignment assertion, cache-identity guard output,
pre-flight loss/VRAM, MICRO_BATCH/ACCUMULATE_STEPS split, and per-epoch
trajectory were not captured/reported.

**best-epoch gold per-finding AUC:**
```
acl_injury                       0.785714
mcl_injury                       0.900000
medial_meniscus_tear             0.402778
lateral_meniscus_tear            0.712121
oa_medial_compartment            1.000000
oa_lateral_compartment                NaN
oa_patellofemoral_compartment    0.583333
effusion                         0.857143
synovitis                        0.757143
bakers_cyst                      1.000000
bone_contusion                   0.600000
fracture                         0.900000
```
**best-epoch gold macro (11-finding): 0.7726**

**SWA gold per-finding AUC:**
```
acl_injury                       0.757143
mcl_injury                       0.933333
medial_meniscus_tear             0.375000
lateral_meniscus_tear            0.712121
oa_medial_compartment            1.000000
oa_lateral_compartment                NaN
oa_patellofemoral_compartment    0.566667
effusion                         0.885714
synovitis                        0.714286
bakers_cyst                      1.000000
bone_contusion                   0.583333
fracture                         0.916667
```
**SWA gold macro (11-finding): 0.7677**

Selection-noise gap (best-epoch minus SWA): **+0.0049** - this project's
first real, on-our-own-data measurement of this gap (see
`feedback_checkpoint_selection_noise.md`).

gold macro (best-epoch, 11-finding) vs. A2 v2 fold-0 baseline 0.7956:
delta=-0.0230, macro_ok=True (tol=0.03).

`medial_meniscus_tear`: 0.4028 (baseline 0.597, delta=-0.1942)
`lateral_meniscus_tear`: 0.7121 (baseline 0.652, delta=+0.0601)
`mcl_injury` (context only, not gating): 0.9000 (baseline 0.8, delta=+0.1000)

At least one meniscus finding moved >=0.10 (pre-agreed I4 bar): **False**

**DECISION: macro OK but neither meniscus finding cleared the noise bar
-> STOP, report as inconclusive, NOT disproved.**

Per the pre-agreed I4 gate (spec section 4): not scaled to 4 folds,
nothing graduated to `src/`. Full write-up in `README.md` History and
project memory.